# Unsupervised Graph Autoencoder (GAE) for BGL Log Anomaly Detection

This notebook adapts the HDFS GAE training pipeline for the **BGL (Blue Gene/L)** dataset.

### Key differences from the HDFS version
| | HDFS | BGL |
|---|---|---|
| **Grouping** | Session (block_id) | Sliding time-window (window_id) |
| **Node features** | 987-dim hybrid (TF-IDF + SBERT) + 9 numeric = 996 | Same structure, same dim |
| **Edge features** | 10-dim (log1p time-deltas + positional) | Same structure — but **near-zero `log1p(td_std)` for most edges** causes BatchNorm blow-up |
| **Data path** | `data/processed/` | `data/processed/BGL/` |
| **Leakage check** | `block_id` attribute | `window_id` attribute |
| **Learning rate** | 1e-2 (safe for HDFS dims) | **1e-3** — TF-IDF sparsity + large node dim requires a smaller LR |
| **Edge normalisation** | In-model `BatchNorm1d(edge_dim)` | **Pre-normalised with clamped global std** — in-model edge BN removed |

### Why the HDFS version would break on BGL
1. **`_is_numeric()` accepted `"nan"` strings** → `float("nan")` does not raise `ValueError`, so NaN values
   entered node feature matrices → `Node loss: nan` from epoch 1.
2. **In-model `BatchNorm1d` on edge features** → BGL's `log1p(td_std)` is 0 for ~99 % of edges
   (events happen near-simultaneously). BatchNorm divides by √(≈0), amplifying non-zero values
   ×100–300. Initial edge MSE ≈ 40 (not ≈ 1), triggering exponential divergence within 5–7 epochs.
3. **Learning rate 1e-2 too high** for 996-dim sparse TF-IDF node features → gradient explosion.

All three issues are fixed in this notebook.

## 1. Imports

In [1]:
import os
import time
import json
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
from torch_geometric.utils import negative_sampling, scatter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, auc,
    f1_score, confusion_matrix, classification_report,
    average_precision_score,
)
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Setup & Configuration

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# BGL: point to the BGL-specific run tag and data directory
RUN_TAG = "20260504_2012"                          # ← update to your BGL run tag
DATA_DIR = Path("../data/processed/BGL")           # BGL lives in its own sub-folder
GRAPH_DATASET_PATH = DATA_DIR / f"{RUN_TAG}_graph_dataset.pt"

# TEST RUN SETTINGS
TEST_RUN    = True    # Set False for full training
TEST_SAMPLES = 5000   # Graphs to use in test run

# Hyperparameters
HIDDEN_DIM    = 128
LATENT_DIM    = 64
BATCH_SIZE    = 256
EPOCHS        = 25
# BGL FIX #3 — lower LR vs HDFS (1e-2 → 1e-3).
# High-dimensional sparse TF-IDF node features (996-dim) cause gradient
# instability at 1e-2; 1e-3 keeps the initial node MSE from exploding.
LEARNING_RATE = 1e-3

# Multi-Task Loss Weights
ALPHA = 1.0   # Structure Reconstruction
BETA  = 1.0   # Node Feature Reconstruction
GAMMA = 1.0   # Edge Attribute Reconstruction

TRAIN_MODE = "clean"   # "clean": normal only | "noisy": mixed

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")
print(f"Test Run Mode: {TEST_RUN} (Samples: {TEST_SAMPLES}, Epochs: {EPOCHS})")


Using device: mps
Test Run Mode: True (Samples: 5000, Epochs: 25)


## 3. Load Dataset & Prepare Splits

In [3]:
print(f"Loading dataset from {GRAPH_DATASET_PATH}...")
dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)

all_data  = dataset["data_list"]
idx_train = dataset["idx_train"]
idx_val   = dataset["idx_val"]
idx_test  = dataset["idx_test"]
NODE_DIM  = dataset["node_dim"]
EDGE_DIM  = dataset["edge_dim"]

print(f"Total Graphs : {len(all_data):,}")
print(f"Node Dim     : {NODE_DIM},  Edge Dim: {EDGE_DIM}")

# ── Filter training set ───────────────────────────────────────────────────────
train_graphs = [all_data[i] for i in idx_train]

if TRAIN_MODE == "clean":
    train_graphs = [g for g in train_graphs if g.y.item() == 0]
    print(f"Clean-Train: training on {len(train_graphs):,} normal graphs.")
else:
    print(f"Noisy-Train: training on all {len(train_graphs):,} graphs.")

if TEST_RUN:
    train_graphs = train_graphs[:TEST_SAMPLES]
    print(f"--- TEST RUN: subsampled to {len(train_graphs)} graphs ---")

val_graphs  = [all_data[i] for i in idx_val]
test_graphs = [all_data[i] for i in idx_test]
if TEST_RUN:
    val_graphs  = val_graphs[:TEST_SAMPLES]
    test_graphs = test_graphs[:TEST_SAMPLES]

# Loaders will be recreated after edge pre-normalisation (next cell)
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE)


Loading dataset from ../data/processed/BGL/20260504_2012_graph_dataset.pt...
Total Graphs : 14,592
Node Dim     : 996,  Edge Dim: 10
Clean-Train: training on 9,182 normal graphs.
--- TEST RUN: subsampled to 5000 graphs ---


## 4. Data Sanity Check

Verifies the dataset contains no NaN/Inf values before training.

> **BGL FIX \#1** — the original `_is_numeric()` helper in the graph-building step accepted
> Python's `float("nan")` and `float("inf")` because they do not raise `ValueError`.
> This caused raw NaN values to enter node feature matrices. The fixed helper (in
> `5_PrepareDataset_BGL.ipynb`) adds `np.isfinite(v)` guard and a final `np.nan_to_num()` sweep.
> This cell confirms the fix is in effect before wasting compute on a broken dataset.

In [4]:
n_nan_nodes  = sum(1 for g in all_data if not torch.isfinite(g.x).all())
n_nan_edges  = sum(1 for g in all_data
                   if g.edge_attr.numel() > 0 and not torch.isfinite(g.edge_attr).all())
n_zero_edges = sum(1 for g in all_data if g.edge_index.size(1) == 0)

print(f"Graphs with NaN/Inf in node features : {n_nan_nodes}")
print(f"Graphs with NaN/Inf in edge features : {n_nan_edges}")
print(f"Graphs with 0 edges (single-event)   : {n_zero_edges:,} "
      f"({100 * n_zero_edges / len(all_data):.1f}%)")

if n_nan_nodes > 0 or n_nan_edges > 0:
    raise ValueError(
        f"Dataset contains NaN/Inf ({n_nan_nodes} node, {n_nan_edges} edge graphs). "
        "Re-run 5_PrepareDataset_BGL.ipynb with the fixed _is_numeric() function."
    )
print("✓ Dataset is clean — no NaN/Inf found.")


Graphs with NaN/Inf in node features : 0
Graphs with NaN/Inf in edge features : 0
Graphs with 0 edges (single-event)   : 2,057 (14.1%)
✓ Dataset is clean — no NaN/Inf found.


## 5. Edge Feature Pre-Normalisation

> **BGL FIX \#2** — root cause of the loss explosion seen in the original BGL attempt.
>
> BGL's `log1p(td_std)` edge feature is **0.0 for ~99 % of edges**: events within a
> sliding window typically happen nearly simultaneously, giving a zero time-delta
> standard deviation. When `BatchNorm1d(eps=1e-5)` encounters a feature that is
> almost always 0, it divides the rare non-zero values by √(≈0), amplifying them
> by 100–300×. This makes the initial edge MSE ≈ 40 instead of ≈ 1, and the
> gradient cascade causes exponential divergence:
> `45 → 138 → 575 → 46 222 → ... → nan`.
>
> **Fix**: compute mean / std from training edges once, clamp std ≥ 0.1 (prevents
> division by near-zero even for degenerate features), then apply z-score
> normalisation to all three splits. The in-model `BatchNorm1d(edge_dim)` is
> **removed** from the model definition (see Section 6) to prevent double-normalisation.
>
> Expected initial edge MSE after this fix: ≈ 1.0.

In [5]:
# ── Compute normalisation statistics from training edges only ─────────────────
ea_list = [g.edge_attr for g in train_graphs if g.edge_attr.numel() > 0]

if ea_list:
    ea_cat    = torch.cat(ea_list, dim=0)          # (total_train_edges, EDGE_DIM)
    EDGE_MEAN = ea_cat.mean(dim=0)
    EDGE_STD  = ea_cat.std(dim=0).clamp(min=0.1)   # clamp prevents division blow-up

    print("Edge feature statistics (before normalisation):")
    print(f"  means : {EDGE_MEAN.numpy().round(3)}")
    print(f"  stds  : {EDGE_STD.numpy().round(3)}")
    print(f"  min std (was BatchNorm instability culprit): {EDGE_STD.min().item():.4f}")

    def _apply_edge_norm(graphs):
        for g in graphs:
            if g.edge_attr.numel() > 0:
                g.edge_attr = (g.edge_attr - EDGE_MEAN) / EDGE_STD

    _apply_edge_norm(train_graphs)
    _apply_edge_norm(val_graphs)
    _apply_edge_norm(test_graphs)

    # Recreate loaders over normalised graphs
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE)

    print(f"\nEdge features pre-normalised for "
          f"{len(train_graphs):,} train / {len(val_graphs):,} val / "
          f"{len(test_graphs):,} test graphs.")
    print("Expected initial edge MSE ≈ 1.0 (was ~40 before fix)")
else:
    EDGE_MEAN = torch.zeros(EDGE_DIM)
    EDGE_STD  = torch.ones(EDGE_DIM)
    print("No edges in training data — edge normalisation skipped.")


Edge feature statistics (before normalisation):
  means : [1.387 0.717 0.803 0.849 0.937 1.168 0.354 0.427 0.483 0.056]
  stds  : [1.24  1.647 1.703 1.741 1.793 1.899 0.997 0.281 0.289 0.131]
  min std (was BatchNorm instability culprit): 0.1310

Edge features pre-normalised for 5,000 train / 2,189 val / 2,189 test graphs.
Expected initial edge MSE ≈ 1.0 (was ~40 before fix)


## 6. Define Attribute-Aware GAE Model

Changes from the HDFS version:
- `self.raw_edge_norm` **removed** — edges are pre-normalised upstream (Section 5).
  Keeping it would cause double-normalisation and re-introduce the blow-up.
- Node BatchNorm (`self.raw_node_norm`) **kept** — node features are high-dimensional
  and sparse (TF-IDF component), so in-model BN on nodes is still beneficial.
- `standardize_inputs` updated accordingly: passes edge features through unchanged.

In [6]:
class AttributeAwareGAE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=128, latent_dim=64):
        super().__init__()

        # --- INPUT STANDARDISATION ---
        # Nodes: BatchNorm tracks running stats → normalises to ~N(0,1).
        # Edges: pre-normalised upstream with global clamped std (Section 5).
        #        No in-model BatchNorm for edges — would re-amplify near-zero features.
        self.raw_node_norm = nn.BatchNorm1d(node_dim, affine=False)
        # (raw_edge_norm intentionally absent — BGL FIX #2)

        # --- ENCODER ---
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        nn1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.encoder_conv = GINEConv(nn1, edge_dim=hidden_dim)

        # --- DECODERS ---
        # 1. Structure: inner product  Z_i · Z_j  (no extra params needed)

        # 2. Node feature decoder
        self.node_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim),
        )

        # 3. Edge attribute decoder
        self.edge_decoder = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_dim),
        )

    def standardize_inputs(self, x, edge_attr):
        x_norm = self.raw_node_norm(x)

        # Edge features already normalised — just ensure shape is correct
        if edge_attr is not None and edge_attr.numel() > 0:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(1)
            edge_attr_norm = edge_attr
        else:
            edge_attr_norm = edge_attr

        return x_norm, edge_attr_norm

    def encode(self, x_norm, edge_index, edge_attr_norm):
        x_h = self.node_proj(x_norm)
        edge_attr_h = self.edge_proj(edge_attr_norm) if edge_attr_norm is not None else None
        return self.encoder_conv(x_h, edge_index, edge_attr_h)

    def decode_structure(self, z, edge_index):
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=1)   # raw logits for BCEWithLogitsLoss

    def decode_node_features(self, z):
        return self.node_decoder(z)

    def decode_edge_attributes(self, z, edge_index):
        src, dst = edge_index
        return self.edge_decoder(torch.cat([z[src], z[dst]], dim=-1))

    def forward(self, x, edge_index, edge_attr):
        x_norm, edge_attr_norm = self.standardize_inputs(x, edge_attr)
        z = self.encode(x_norm, edge_index, edge_attr_norm)
        return z, x_norm, edge_attr_norm


## 7. Multi-Task Training Loop

In [7]:
def train_step(model, loader, optimizer):
    model.train()
    total_loss = total_str = total_node = total_edge = 0.0

    for batch in tqdm(loader, desc="Training", leave=False):
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)

        # 1. Structure loss — guard against all-zero-edge batches.
        #    F.binary_cross_entropy_with_logits on an empty tensor returns nan
        #    (sum([]) / 0), which poisons all gradients via backprop.
        loss_str = torch.tensor(0.0, device=DEVICE)
        if batch.edge_index.size(1) > 0:
            pos_logits = model.decode_structure(z, batch.edge_index)
            neg_edge   = negative_sampling(
                batch.edge_index,
                num_nodes=batch.num_nodes,
                num_neg_samples=batch.edge_index.size(1),
            )
            neg_logits = model.decode_structure(z, neg_edge)
            loss_str   = (
                F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
                + F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
            )

        # 2. Node feature loss
        x_rec     = model.decode_node_features(z)
        loss_node = F.mse_loss(x_rec, x_norm)

        # 3. Edge attribute loss
        loss_edge = torch.tensor(0.0, device=DEVICE)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            edge_rec  = model.decode_edge_attributes(z, batch.edge_index)
            loss_edge = F.mse_loss(edge_rec, edge_attr_norm)

        loss = ALPHA * loss_str + BETA * loss_node + GAMMA * loss_edge
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n = batch.num_graphs
        total_loss  += loss.item()  * n
        total_str   += loss_str.item()  * n
        total_node  += loss_node.item() * n
        total_edge  += loss_edge.item() * n

    ng = len(loader.dataset)
    if ng == 0:
        return 0.0, 0.0, 0.0, 0.0
    return total_loss/ng, total_str/ng, total_node/ng, total_edge/ng


# ── Initialise & train ────────────────────────────────────────────────────────
model     = AttributeAwareGAE(NODE_DIM, EDGE_DIM, HIDDEN_DIM, LATENT_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
history   = {"total": [], "structure": [], "node": [], "edge": []}

print("Starting Training...")
print(f"Weights → ALPHA (Str): {ALPHA}, BETA (Node): {BETA}, GAMMA (Edge): {GAMMA}")
print(f"Expected epoch 1 losses: Str ≈ 1.3, Node ≈ 1.0, Edge ≈ 1.0")

for epoch in range(1, EPOCHS + 1):
    l_tot, l_str, l_nod, l_edg = train_step(model, train_loader, optimizer)
    history["total"].append(l_tot)
    history["structure"].append(l_str)
    history["node"].append(l_nod)
    history["edge"].append(l_edg)
    print(f"Epoch {epoch:02d} | Loss: {l_tot:.4f} "
          f"(Str: {l_str:.4f}, Node: {l_nod:.4f}, Edge: {l_edg:.4f})")


Starting Training...
Weights → ALPHA (Str): 1.0, BETA (Node): 1.0, GAMMA (Edge): 1.0
Expected epoch 1 losses: Str ≈ 1.3, Node ≈ 1.0, Edge ≈ 1.0


Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | Loss: 2.7718 (Str: 1.2150, Node: 0.6518, Edge: 0.9050)


Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 02 | Loss: 2.1799 (Str: 0.9419, Node: 0.6111, Edge: 0.6269)


...

Epoch 06 | Loss: 1.5353 (Str: 0.9103, Node: 0.3358, Edge: 0.2891)


Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 07 | Loss: 1.4601 (Str: 0.8937, Node: 0.2965, Edge: 0.2699)


Training:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Inference & Validation

In [ ]:
def evaluate_anomaly_scores(model, loader):
    model.eval()
    all_scores, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = batch.to(DEVICE)
            z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
            num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1

            # Structure error
            graph_str_error = torch.zeros(num_graphs, device=DEVICE)
            if batch.edge_index.size(1) > 0:
                pos_logits = model.decode_structure(z, batch.edge_index)
                pos_probs  = torch.sigmoid(pos_logits)
                edge_err   = F.binary_cross_entropy(
                    pos_probs, torch.ones_like(pos_probs), reduction="none"
                )
                edge_batch = batch.batch[batch.edge_index[0]]
                graph_str_error = scatter(
                    edge_err, edge_batch, dim=0, reduce="mean", dim_size=num_graphs
                )

            # Node error
            x_rec        = model.decode_node_features(z)
            node_errors  = F.mse_loss(x_rec, x_norm, reduction="none").mean(dim=1)
            batch_idx    = batch.batch
            graph_node_error = scatter(
                node_errors, batch_idx, dim=0, reduce="mean", dim_size=num_graphs
            )

            # Edge error
            graph_edge_error = torch.zeros(num_graphs, device=DEVICE)
            if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
                edge_rec    = model.decode_edge_attributes(z, batch.edge_index)
                ea_errors   = F.mse_loss(edge_rec, edge_attr_norm, reduction="none").mean(dim=1)
                edge_batch  = batch.batch[batch.edge_index[0]]
                graph_edge_error = scatter(
                    ea_errors, edge_batch, dim=0, reduce="mean", dim_size=num_graphs
                )

            graph_str_error  = torch.nan_to_num(graph_str_error,  0.0)
            graph_edge_error = torch.nan_to_num(graph_edge_error, 0.0)

            total = ALPHA * graph_str_error + BETA * graph_node_error + GAMMA * graph_edge_error
            all_scores.append(total.cpu())
            all_labels.append(batch.y.cpu())

    return torch.cat(all_scores).numpy(), torch.cat(all_labels).numpy()


print("Evaluating on Validation Set...")
val_scores, val_labels = evaluate_anomaly_scores(model, val_loader)

precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)
f1_scores   = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx    = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nValidation Complete!")
print(f"Best Threshold  : {best_threshold:.4f}")
print(f"Best Val F1     : {f1_scores[best_idx]:.4f}")
print(f"Val PR-AUC      : {auc(recall, precision):.4f}")
print(f"Val ROC-AUC     : {roc_auc_score(val_labels, val_scores):.4f}")


## 9. Test Evaluation

In [ ]:
print("Evaluating on Test Set...")
test_scores, test_labels = evaluate_anomaly_scores(model, test_loader)

test_preds  = (test_scores > best_threshold).astype(int)

print("\n=== Test Set Results ===")
print(classification_report(test_labels, test_preds, digits=4))

test_pr_auc  = auc(*precision_recall_curve(test_labels, test_scores)[1::-1])
test_roc_auc = roc_auc_score(test_labels, test_scores)
print(f"Test PR-AUC  : {test_pr_auc:.4f}")
print(f"Test ROC-AUC : {test_roc_auc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(test_scores[test_labels == 0], color="steelblue",
             label="Normal",  stat="density", alpha=0.5, bins=50, ax=axes[0])
sns.histplot(test_scores[test_labels == 1], color="firebrick",
             label="Anomaly", stat="density", alpha=0.5, bins=50, ax=axes[0])
axes[0].axvline(best_threshold, color="black", linestyle="--", label="Threshold")
axes[0].set_title("Anomaly Score Distribution")
axes[0].set_xlabel("Reconstruction Error")
axes[0].legend()

cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 9b. Load a Previously Trained Model (Optional)

Run this cell *instead of* Section 7 (Training) to reload a checkpoint. Sections 1–6 must have been executed first.

In [ ]:
# # ── Load a saved model checkpoint ─────────────────────────────────────────────
# MODEL_DIR  = Path("../models/")
# MODEL_PATH = MODEL_DIR / f"{RUN_TAG}_bgl_attribute_gae.pt"
#
# print(f"Loading model from {MODEL_PATH}...")
# checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
#
# model = AttributeAwareGAE(
#     node_dim   = checkpoint["node_dim"],
#     edge_dim   = checkpoint["edge_dim"],
#     hidden_dim = checkpoint["hidden_dim"],
#     latent_dim = checkpoint["latent_dim"],
# ).to(DEVICE)
#
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()
#
# best_threshold = checkpoint["best_threshold"]
# history        = checkpoint.get("history", None)
# EDGE_MEAN      = checkpoint["edge_mean"]
# EDGE_STD       = checkpoint["edge_std"]
#
# # Re-apply edge normalisation to val/test graphs using saved statistics
# def _apply_edge_norm(graphs):
#     for g in graphs:
#         if g.edge_attr.numel() > 0:
#             g.edge_attr = (g.edge_attr - EDGE_MEAN) / EDGE_STD
#
# _apply_edge_norm(val_graphs)
# _apply_edge_norm(test_graphs)
#
# val_loader  = DataLoader(val_graphs,  batch_size=BATCH_SIZE)
# test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)
#
# print(f"Model loaded. Best Threshold: {best_threshold:.4f}")


## 10. Training Loss Curves

In [ ]:
if history is None:
    print("No training history available. Train the model first (Section 7).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs_r  = range(1, len(history["total"]) + 1)

    axes[0].plot(epochs_r, history["total"], "k-", linewidth=2)
    axes[0].set_title("Total Training Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_r, history["structure"], "b-", lw=2, label="Structure (BCE)")
    axes[1].plot(epochs_r, history["node"],      "g-", lw=2, label="Node (MSE)")
    axes[1].plot(epochs_r, history["edge"],      "r-", lw=2, label="Edge (MSE)")
    axes[1].set_title("Component Losses (Absolute)")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    total    = np.array(history["total"])
    str_pct  = np.array(history["structure"]) / total * 100
    node_pct = np.array(history["node"])      / total * 100
    edge_pct = np.array(history["edge"])      / total * 100
    axes[2].stackplot(epochs_r, str_pct, node_pct, edge_pct,
                      labels=["Structure", "Node Features", "Edge Attributes"],
                      colors=["#4C72B0", "#55A868", "#C44E52"], alpha=0.8)
    axes[2].set_title("Loss Component Proportion (%)")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("% of Total Loss")
    axes[2].set_ylim(0, 100); axes[2].legend(loc="center right")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 11. Component Decomposition of Anomaly Scores

In [ ]:
def evaluate_component_scores(model, loader):
    model.eval()
    all_str, all_node, all_edge, all_labels = [], [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Component Scoring", leave=False):
            batch = batch.to(DEVICE)
            z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
            num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1

            # Structure
            g_str = torch.zeros(num_graphs, device=DEVICE)
            if batch.edge_index.size(1) > 0:
                pos_probs  = torch.sigmoid(model.decode_structure(z, batch.edge_index))
                edge_err   = F.binary_cross_entropy(
                    pos_probs, torch.ones_like(pos_probs), reduction="none"
                )
                edge_batch = batch.batch[batch.edge_index[0]]
                g_str      = scatter(edge_err, edge_batch, dim=0, reduce="mean", dim_size=num_graphs)

            # Node
            x_rec   = model.decode_node_features(z)
            ne      = F.mse_loss(x_rec, x_norm, reduction="none").mean(dim=1)
            g_node  = scatter(ne, batch.batch, dim=0, reduce="mean", dim_size=num_graphs)

            # Edge
            g_edge = torch.zeros(num_graphs, device=DEVICE)
            if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
                ea_err = F.mse_loss(
                    model.decode_edge_attributes(z, batch.edge_index),
                    edge_attr_norm, reduction="none"
                ).mean(dim=1)
                edge_batch = batch.batch[batch.edge_index[0]]
                g_edge = scatter(ea_err, edge_batch, dim=0, reduce="mean", dim_size=num_graphs)

            all_str.append(torch.nan_to_num(g_str,  0.0).cpu())
            all_node.append(g_node.cpu())
            all_edge.append(torch.nan_to_num(g_edge, 0.0).cpu())
            all_labels.append(batch.y.cpu())

    return (torch.cat(all_str).numpy(), torch.cat(all_node).numpy(),
            torch.cat(all_edge).numpy(), torch.cat(all_labels).numpy())


test_str, test_node, test_edge, test_y = evaluate_component_scores(model, test_loader)
print(f"Component scores computed for {len(test_y):,} test graphs.")

# ── Per-component distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
components = [
    ("Structure Error",       test_str),
    ("Node Feature Error",    test_node),
    ("Edge Attribute Error",  test_edge),
]
for ax, (name, scores), _ in zip(axes, components, range(3)):
    sns.histplot(scores[test_y == 0], color="steelblue",
                 label=f"Normal (n={int((test_y==0).sum())})",
                 stat="density", alpha=0.5, bins=50, ax=ax)
    sns.histplot(scores[test_y == 1], color="firebrick",
                 label=f"Anomaly (n={int((test_y==1).sum())})",
                 stat="density", alpha=0.5, bins=50, ax=ax)
    ax.set_title(name); ax.set_xlabel("Reconstruction Error")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("Per-Component Score Distributions: Normal vs Anomaly", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 12. Precision-Recall & ROC Curves

In [ ]:
combined_scores = test_str + test_node + test_edge

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve
prec, rec, _ = precision_recall_curve(test_y, combined_scores)
pr_auc_val   = auc(rec, prec)
axes[0].plot(rec, prec, "b-", lw=2, label=f"PR Curve (AUC = {pr_auc_val:.4f})")
axes[0].fill_between(rec, prec, alpha=0.1, color="blue")
baseline = test_y.sum() / len(test_y)
axes[0].axhline(baseline, color="gray", linestyle="--",
                label=f"Random Baseline ({baseline:.3f})")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve",
            xlim=[0, 1], ylim=[0, 1.05])
axes[0].legend(loc="lower left"); axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr, tpr, _ = roc_curve(test_y, combined_scores)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, "r-", lw=2, label=f"ROC Curve (AUC = {roc_auc_val:.4f})")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="red")
axes[1].plot([0, 1], [0, 1], "gray", linestyle="--", label="Random Baseline")
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve",
            xlim=[0, 1], ylim=[0, 1.05])
axes[1].legend(loc="lower right"); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 13. Component Contribution Analysis

For each correctly detected anomaly, which reconstruction component drove the score?

In [ ]:
test_preds = (combined_scores > best_threshold).astype(int)

tp_mask = (test_y == 1) & (test_preds == 1)
fn_mask = (test_y == 1) & (test_preds == 0)

if tp_mask.sum() > 0:
    tp_str, tp_node, tp_edge = test_str[tp_mask], test_node[tp_mask], test_edge[tp_mask]
    tp_total   = tp_str + tp_node + tp_edge + 1e-10
    tp_str_pct  = tp_str  / tp_total * 100
    tp_node_pct = tp_node / tp_total * 100
    tp_edge_pct = tp_edge / tp_total * 100

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    wedge_colors = ["#4C72B0", "#55A868", "#C44E52"]

    axes[0].pie(
        [tp_str_pct.mean(), tp_node_pct.mean(), tp_edge_pct.mean()],
        labels=["Structure", "Node Features", "Edge Attributes"],
        autopct="%1.1f%%", colors=wedge_colors, startangle=90,
    )
    axes[0].set_title(f"Avg. Contribution to Detected Anomalies\n(n={tp_mask.sum()})")

    dominant = np.argmax(np.column_stack([tp_str, tp_node, tp_edge]), axis=1)
    dom_counts = [np.sum(dominant == i) for i in range(3)]
    bars = axes[1].bar(
        ["Structure", "Node Features", "Edge Attributes"], dom_counts,
        color=wedge_colors, edgecolor="black", alpha=0.85,
    )
    axes[1].set_title(f"Dominant Component per Detected Anomaly\n(n={tp_mask.sum()})")
    axes[1].set_ylabel("Count"); axes[1].grid(True, alpha=0.3, axis="y")
    for bar, v in zip(bars, dom_counts):
        axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
                     ha="center", fontweight="bold")

    df_box = pd.DataFrame({
        "Structure":       np.concatenate([test_str[test_y==0],  test_str[test_y==1]]),
        "Node Features":   np.concatenate([test_node[test_y==0], test_node[test_y==1]]),
        "Edge Attributes": np.concatenate([test_edge[test_y==0], test_edge[test_y==1]]),
        "Label": ["Normal"]*(test_y==0).sum() + ["Anomaly"]*(test_y==1).sum(),
    })
    sns.boxplot(data=df_box.melt(id_vars="Label", var_name="Component", value_name="Error"),
                x="Component", y="Error", hue="Label",
                palette={"Normal": "steelblue", "Anomaly": "firebrick"}, ax=axes[2])
    axes[2].set_title("Error Distribution by Component & Label")
    axes[2].set_ylabel("Reconstruction Error"); axes[2].grid(True, alpha=0.3, axis="y")

    plt.tight_layout(); plt.show()

    print(f"\n--- Detected Anomalies (True Positives): {tp_mask.sum()} ---")
    print(f"  Avg Structure Error : {tp_str.mean():.4f}  ({tp_str_pct.mean():.1f}%)")
    print(f"  Avg Node Error      : {tp_node.mean():.4f}  ({tp_node_pct.mean():.1f}%)")
    print(f"  Avg Edge Error      : {tp_edge.mean():.4f}  ({tp_edge_pct.mean():.1f}%)")
else:
    print("No true positives found — lower the threshold or train for more epochs.")

print(f"\n--- Missed Anomalies (False Negatives): {fn_mask.sum()} ---")
if fn_mask.sum() > 0:
    fn_total = test_str[fn_mask] + test_node[fn_mask] + test_edge[fn_mask]
    print(f"  Avg Total Score     : {fn_total.mean():.4f}  (threshold: {best_threshold:.4f})")


## 14. Data Leakage Verification

BGL uses `window_id` (not HDFS `block_id`) as the graph identifier. The overlap check is updated accordingly.

In [ ]:
# BGL graphs carry window_id instead of block_id
# Retrieve window_id gracefully — older checkpoints may not have it
def _get_wid(g):
    return getattr(g, "window_id", getattr(g, "block_id", id(g)))

train_wids = set(_get_wid(g) for g in train_graphs)
val_wids   = set(_get_wid(g) for g in val_graphs)
test_wids  = set(_get_wid(g) for g in test_graphs)

overlap_tv = train_wids & val_wids
overlap_tt = train_wids & test_wids
overlap_vt = val_wids   & test_wids

print("=== Data Leakage Verification ===")
print(f"Train window_ids : {len(train_wids):,}")
print(f"Val   window_ids : {len(val_wids):,}")
print(f"Test  window_ids : {len(test_wids):,}")
print(f"\nOverlap Train-Val  : {len(overlap_tv)} {'PASS' if not overlap_tv else 'FAIL'}")
print(f"Overlap Train-Test : {len(overlap_tt)} {'PASS' if not overlap_tt else 'FAIL'}")
print(f"Overlap Val-Test   : {len(overlap_vt)} {'PASS' if not overlap_vt else 'FAIL'}")

train_lbl = np.array([g.y.item() for g in train_graphs])
val_lbl   = np.array([g.y.item() for g in val_graphs])
test_lbl  = np.array([g.y.item() for g in test_graphs])

print(f"\n--- Label Distribution ---")
print(f"Train : {len(train_lbl):,} graphs, anomaly rate: {train_lbl.mean():.4f} ({train_lbl.sum():,})")
print(f"Val   : {len(val_lbl):,} graphs, anomaly rate: {val_lbl.mean():.4f} ({val_lbl.sum():,})")
print(f"Test  : {len(test_lbl):,} graphs, anomaly rate: {test_lbl.mean():.4f} ({test_lbl.sum():,})")

if TRAIN_MODE == "clean":
    assert train_lbl.sum() == 0, "FAIL: anomalous graphs found in clean training set!"
    print("\nClean-Train: 0 anomalies in training set. PASS")
else:
    print("\nNoisy-Train: labels never used during training. PASS")

print("\n--- BatchNorm Running Stats (node normaliser) ---")
print(f"  running_mean (first 5): {model.raw_node_norm.running_mean[:5].cpu().numpy()}")
print(f"  running_var  (first 5): {model.raw_node_norm.running_var[:5].cpu().numpy()}")


## 15. Save Model

In [ ]:
MODEL_DIR  = Path("../models/")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / f"{RUN_TAG}_bgl_attribute_gae.pt"

checkpoint = {
    "node_dim"         : NODE_DIM,
    "edge_dim"         : EDGE_DIM,
    "hidden_dim"       : HIDDEN_DIM,
    "latent_dim"       : LATENT_DIM,
    "model_state_dict" : model.state_dict(),
    "best_threshold"   : best_threshold,
    "history"          : history,
    # Save edge normalisation statistics so the model can be reloaded
    # and applied to new data without re-running the training set pass.
    "edge_mean"        : EDGE_MEAN,
    "edge_std"         : EDGE_STD,
}

torch.save(checkpoint, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")
print(f"  node_dim={NODE_DIM}, edge_dim={EDGE_DIM}, "
      f"hidden_dim={HIDDEN_DIM}, latent_dim={LATENT_DIM}")
print(f"  best_threshold={best_threshold:.4f}")
print("  edge_mean and edge_std saved for checkpoint reloading (Section 9b).")
